# 02 — Baseline Models with Rolling Time-Window CV

**Project**: UPI Fraud Detection — Drift-Aware Explainable ML Framework
**Goal**: Train Logistic Regression, Random Forest, XGBoost, LightGBM with rolling time-window CV
**Metrics**: PR-AUC, Precision@K, Recall@K, ROC-AUC

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import json

from data_prep import load_ieee_cis, load_paysim, prepare_features_target, handle_class_imbalance
from models import get_model, train_model, evaluate_model, rolling_time_cv, train_all_baselines, save_model

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_ROOT = Path('../data')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

print(f"Data root: {DATA_ROOT.absolute()}")

## 1. Load Pre-Split Data

In [ ]:
# Load IEEE-CIS splits
ieee_split_dir = DATA_ROOT / 'ieee-cis' / 'splits'
try:
    train_ieee = pd.read_parquet(ieee_split_dir / 'train.parquet')
    val_ieee = pd.read_parquet(ieee_split_dir / 'val.parquet')
    test_ieee = pd.read_parquet(ieee_split_dir / 'test.parquet')
    print(f"IEEE-CIS splits loaded:")
    print(f"  Train: {train_ieee.shape}")
    print(f"  Val: {val_ieee.shape}")
    print(f"  Test: {test_ieee.shape}")
except FileNotFoundError:
    print("IEEE-CIS splits not found. Run 01_eda.ipynb first.")
    train_ieee = val_ieee = test_ieee = None

In [ ]:
# Load PaySim splits
paysim_split_dir = DATA_ROOT / 'paysim' / 'splits'
try:
    train_paysim = pd.read_parquet(paysim_split_dir / 'train.parquet')
    val_paysim = pd.read_parquet(paysim_split_dir / 'val.parquet')
    test_paysim = pd.read_parquet(paysim_split_dir / 'test.parquet')
    print(f"PaySim splits loaded:")
    print(f"  Train: {train_paysim.shape}")
    print(f"  Val: {val_paysim.shape}")
    print(f"  Test: {test_paysim.shape}")
except FileNotFoundError:
    print("PaySim splits not found. Run 01_eda.ipynb first.")
    train_paysim = val_paysim = test_paysim = None

## 2. Feature Engineering

In [ ]:
def apply_feature_engineering(df, dataset_name):
    """Apply consistent feature engineering."""
    df = df.copy()
    
    if dataset_name == 'ieee-cis':
        df['hour'] = (df['TransactionDT'] // 3600) % 24
        df['day_of_week'] = (df['TransactionDT'] // 86400) % 7
        df['amount_log'] = np.log1p(df['TransactionAmt'])
        df['amount_round'] = (df['TransactionAmt'] == df['TransactionAmt'].round()).astype(int)
        if 'card1' in df.columns:
            df['card1_freq'] = df.groupby('card1')['card1'].transform('count')
    
    elif dataset_name == 'paysim':
        df['hour'] = df['step'] % 24
        df['day'] = df['step'] // 24
        df['amount_log'] = np.log1p(df['amount'])
        df['amount_round'] = (df['amount'] == df['amount'].round()).astype(int)
        df['orig_balance_diff'] = df['oldbalanceOrg'] - df['newbalanceOrig']
        df['dest_balance_diff'] = df['newbalanceDest'] - df['oldbalanceDest']
        df['zero_balance_orig'] = (df['oldbalanceOrg'] == 0).astype(int)
        df['zero_balance_dest'] = (df['oldbalanceDest'] == 0).astype(int)
    
    return df

if train_ieee is not None:
    train_ieee = apply_feature_engineering(train_ieee, 'ieee-cis')
    val_ieee = apply_feature_engineering(val_ieee, 'ieee-cis')
    test_ieee = apply_feature_engineering(test_ieee, 'ieee-cis')
    print(f"IEEE-CIS features: {train_ieee.shape[1]}")

if train_paysim is not None:
    train_paysim = apply_feature_engineering(train_paysim, 'paysim')
    val_paysim = apply_feature_engineering(val_paysim, 'paysim')
    test_paysim = apply_feature_engineering(test_paysim, 'paysim')
    print(f"PaySim features: {train_paysim.shape[1]}")

In [ ]:
def prepare_xy(df, dataset_name):
    """Prepare X, y with dataset-specific drops."""
    if dataset_name == 'ieee-cis':
        drop_cols = ['TransactionID', 'TransactionDT', 'isFraud']
    else:
        drop_cols = ['step', 'isFraud', 'isFlaggedFraud', 'nameOrig', 'nameDest']
    
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df['isFraud'].astype('int8')
    
    for col in X.select_dtypes(include=['category']).columns:
        X[col] = X[col].cat.codes
    
    X = X.fillna(-999)
    
    return X, y

if train_ieee is not None:
    X_train_ieee, y_train_ieee = prepare_xy(train_ieee, 'ieee-cis')
    X_val_ieee, y_val_ieee = prepare_xy(val_ieee, 'ieee-cis')
    X_test_ieee, y_test_ieee = prepare_xy(test_ieee, 'ieee-cis')
    print(f"IEEE-CIS X_train: {X_train_ieee.shape}")

if train_paysim is not None:
    X_train_p, y_train_p = prepare_xy(train_paysim, 'paysim')
    X_val_p, y_val_p = prepare_xy(val_paysim, 'paysim')
    X_test_p, y_test_p = prepare_xy(test_paysim, 'paysim')
    print(f"PaySim X_train: {X_train_p.shape}")

## 3. Class Imbalance Parameters

In [ ]:
if train_ieee is not None:
    ieee_params = handle_class_imbalance(X_train_ieee, y_train_ieee, method='class_weight')
    print(f"IEEE-CIS imbalance params: {ieee_params}")

if train_paysim is not None:
    paysim_params = handle_class_imbalance(X_train_p, y_train_p, method='class_weight')
    print(f"PaySim imbalance params: {paysim_params}")

## 4. Train All Baselines on IEEE-CIS

In [ ]:
if train_ieee is not None:
    print("="*60)
    print("TRAINING BASELINES ON IEEE-CIS")
    print("="*60)
    
    ieee_results = train_all_baselines(
        X_train_ieee, y_train_ieee,
        X_val_ieee, y_val_ieee,
        class_weight_params=ieee_params,
        time_col='TransactionDT'
    )
    
    for name, result in ieee_results.items():
        save_model(result['model'], MODELS_DIR / f'ieee_{name}.joblib',
                  {'model': name, 'dataset': 'ieee-cis', 'metrics': result['val_metrics']})
    
    metrics_df = pd.DataFrame({name: r['val_metrics'] for name, r in ieee_results.items()}).T
    metrics_df.to_csv(REPORTS_DIR / 'ieee_baseline_val_metrics.csv')
    print(f"\nValidation metrics saved to {REPORTS_DIR / 'ieee_baseline_val_metrics.csv'}")
    display(metrics_df[['pr_auc', 'roc_auc', 'precision_at_100', 'recall_at_100']])

## 5. Train All Baselines on PaySim

In [ ]:
if train_paysim is not None:
    print("="*60)
    print("TRAINING BASELINES ON PAYSIM")
    print("="*60)
    
    paysim_results = train_all_baselines(
        X_train_p, y_train_p,
        X_val_p, y_val_p,
        class_weight_params=paysim_params,
        time_col='step'
    )
    
    for name, result in paysim_results.items():
        save_model(result['model'], MODELS_DIR / f'paysim_{name}.joblib',
                  {'model': name, 'dataset': 'paysim', 'metrics': result['val_metrics']})
    
    metrics_df_p = pd.DataFrame({name: r['val_metrics'] for name, r in paysim_results.items()}).T
    metrics_df_p.to_csv(REPORTS_DIR / 'paysim_baseline_val_metrics.csv')
    print(f"\nValidation metrics saved to {REPORTS_DIR / 'paysim_baseline_val_metrics.csv'}")
    display(metrics_df_p[['pr_auc', 'roc_auc', 'precision_at_100', 'recall_at_100']])

## 6. Rolling Time-Window Cross-Validation

In [ ]:
if train_ieee is not None:
    print("="*60)
    print("ROLLING TIME-WINDOW CV ON IEEE-CIS")
    print("="*60)
    
    cv_results_ieee = {}
    for model_name in ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm']:
        print(f"\n{model_name}:")
        cv_results_ieee[model_name] = rolling_time_cv(
            X_train_ieee, y_train_ieee,
            model_name,
            n_splits=5,
            train_window=30,
            test_window=7,
            time_col='TransactionDT',
            class_weight_params=ieee_params
        )

In [ ]:
if train_paysim is not None:
    print("="*60)
    print("ROLLING TIME-WINDOW CV ON PAYSIM")
    print("="*60)
    
    cv_results_paysim = {}
    for model_name in ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm']:
        print(f"\n{model_name}:")
        cv_results_paysim[model_name] = rolling_time_cv(
            X_train_p, y_train_p,
            model_name,
            n_splits=5,
            train_window=30,
            test_window=7,
            time_col='step',
            class_weight_params=paysim_params
        )

## 7. CV Results Summary

In [ ]:
def summarize_cv(cv_results, dataset_name):
    """Summarize CV results."""
    summary = []
    for model_name, folds in cv_results.items():
        if not folds:
            continue
        df = pd.DataFrame(folds)
        summary.append({
            'model': model_name,
            'dataset': dataset_name,
            'pr_auc_mean': df['pr_auc'].mean(),
            'pr_auc_std': df['pr_auc'].std(),
            'roc_auc_mean': df['roc_auc'].mean(),
            'p100_mean': df.get('precision_at_100', pd.Series([0])).mean(),
            'r100_mean': df.get('recall_at_100', pd.Series([0])).mean(),
        })
    return pd.DataFrame(summary)

if train_ieee is not None:
    cv_summary_ieee = summarize_cv(cv_results_ieee, 'ieee-cis')
    print("IEEE-CIS CV Summary:")
    display(cv_summary_ieee)
    cv_summary_ieee.to_csv(REPORTS_DIR / 'ieee_cv_summary.csv', index=False)

if train_paysim is not None:
    cv_summary_paysim = summarize_cv(cv_results_paysim, 'paysim')
    print("PaySim CV Summary:")
    display(cv_summary_paysim)
    cv_summary_paysim.to_csv(REPORTS_DIR / 'paysim_cv_summary.csv', index=False)

## 8. Test Set Evaluation (Final)

In [ ]:
if train_ieee is not None:
    print("="*60)
    print("TEST SET EVALUATION - IEEE-CIS")
    print("="*60)
    
    test_results_ieee = {}
    for name, result in ieee_results.items():
        metrics = evaluate_model(result['model'], X_test_ieee, y_test_ieee)
        test_results_ieee[name] = metrics
        print(f"{name}: PR-AUC={metrics['pr_auc']:.4f}, P@100={metrics.get('precision_at_100', 0):.4f}, R@100={metrics.get('recall_at_100', 0):.4f}")
    
    test_df_ieee = pd.DataFrame(test_results_ieee).T
    test_df_ieee.to_csv(REPORTS_DIR / 'ieee_test_metrics.csv')
    display(test_df_ieee[['pr_auc', 'roc_auc', 'precision_at_100', 'recall_at_100']])

In [ ]:
if train_paysim is not None:
    print("="*60)
    print("TEST SET EVALUATION - PAYSIM")
    print("="*60)
    
    test_results_paysim = {}
    for name, result in paysim_results.items():
        metrics = evaluate_model(result['model'], X_test_p, y_test_p)
        test_results_paysim[name] = metrics
        print(f"{name}: PR-AUC={metrics['pr_auc']:.4f}, P@100={metrics.get('precision_at_100', 0):.4f}, R@100={metrics.get('recall_at_100', 0):.4f}")
    
    test_df_paysim = pd.DataFrame(test_results_paysim).T
    test_df_paysim.to_csv(REPORTS_DIR / 'paysim_test_metrics.csv')
    display(test_df_paysim[['pr_auc', 'roc_auc', 'precision_at_100', 'recall_at_100']])

## 9. Visualization: Model Comparison

In [ ]:
def plot_model_comparison(test_results_ieee, test_results_paysim):
    """Plot model comparison across datasets."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    metrics_to_plot = ['pr_auc', 'roc_auc', 'precision_at_100', 'recall_at_100']
    titles = ['PR-AUC', 'ROC-AUC', 'Precision@100', 'Recall@100']
    
    for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
        ax = axes[idx // 2, idx % 2]
        
        if test_results_ieee:
            ieee_vals = [test_results_ieee[m].get(metric, 0) for m in ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm']]
            ax.bar([f'{m}\n(IEEE)' for m in ['LR', 'RF', 'XGB', 'LGB']], ieee_vals, alpha=0.7, label='IEEE-CIS', color='steelblue')
        
        if test_results_paysim:
            paysim_vals = [test_results_paysim[m].get(metric, 0) for m in ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm']]
            x_pos = np.arange(4) + 0.35
            ax.bar(x_pos, paysim_vals, alpha=0.7, label='PaySim', width=0.35, color='orange')
        
        ax.set_title(title)
        ax.set_ylabel('Score')
        ax.legend()
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

if train_ieee is not None and train_paysim is not None:
    plot_model_comparison(test_results_ieee, test_results_paysim)

## 10. Summary & Next Steps

In [ ]:
print("""
=== BASELINE TRAINING COMPLETE ===

✅ Models trained: Logistic Regression, Random Forest, XGBoost, LightGBM
✅ Rolling time-window CV completed (5 folds)
✅ Test set evaluation done
✅ Models saved to ../models/
✅ Metrics saved to ../reports/

NEXT: Notebook 03_delayed_labels.ipynb
- Simulate delayed labels (1/7/30-day)
- Rolling window evaluation with delayed ground truth
- Compare vs oracle (no delay)
""")